# 02 — Data Cleaning

This notebook prepares clean and consistent versions of the Olist datasets for the next stages of the SupplyGuard project.

The previous notebook focused on understanding the raw data structure, identifying table relationships, missing values, datetime-like columns, and potential leakage risks. This notebook uses those findings to apply a conservative cleaning process without changing the analytical meaning of the data.

The goal is not to create the final machine learning dataset yet. Instead, this notebook creates a reliable processed data layer that can be safely reused for relational modeling, SQL analysis, EDA, feature engineering, dashboarding, and later predictive modeling.

The cleaning process focuses on:

- loading the validated raw datasets;
- converting relevant date and timestamp columns;
- reviewing and removing exact duplicate rows where appropriate;
- documenting missing values without aggressive imputation;
- applying conservative text cleaning;
- enriching product categories with English translations;
- creating an aggregated geolocation table by zip code prefix to support safe geographic joins;
- preserving leakage-sensitive variables for future target creation and historical analysis.

This notebook intentionally avoids heavy feature engineering, visual EDA, model training, and final target creation. Those steps belong to later stages of the project.

Important leakage note: post-delivery variables are kept in the cleaned tables because they are needed for target definition and historical analysis. However, they must not be used directly as future machine learning features.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.api.types import is_object_dtype, is_string_dtype


from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 120)

## 1. Imports, Paths and Expected Files

The project uses relative paths with `pathlib` so the notebook can run without hardcoded local paths.

In [5]:
expected_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}


def find_project_root(start_path=None, expected_raw_files=None):
    start_path = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    expected_raw_files = expected_raw_files or {}

    for path in [start_path, *start_path.parents]:
        raw_dir = path / "data" / "raw"

        if raw_dir.exists() and all((raw_dir / file_name).exists() for file_name in expected_raw_files.values()):
            return path

    raise FileNotFoundError(
        "Project root could not be detected. Make sure this notebook is inside the project folder "
        "and that data/raw/ contains all expected Olist CSV files."
    )


PROJECT_ROOT = find_project_root(expected_raw_files=expected_files)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")

Project root: C:\Users\johan\Desktop\supplyguard-delivery-risk
Raw data directory: C:\Users\johan\Desktop\supplyguard-delivery-risk\data\raw
Processed data directory: C:\Users\johan\Desktop\supplyguard-delivery-risk\data\processed


In [6]:
file_check_df = pd.DataFrame([
    {"table_name": table_name, "expected_file": file_name, "file_found": (RAW_DIR / file_name).exists()}
    for table_name, file_name in expected_files.items()
])

display(file_check_df)

missing_files = file_check_df.loc[~file_check_df["file_found"], "expected_file"].tolist()

if missing_files:
    raise FileNotFoundError(f"Missing raw files: {missing_files}")

,table_name,expected_file,file_found
0,customers,olist_customers_dataset.csv,True
1,geolocation,olist_geolocation_dataset.csv,True
2,order_items,olist_order_items_dataset.csv,True
3,order_payments,olist_order_payments_dataset.csv,True
4,order_reviews,olist_order_reviews_dataset.csv,True
5,orders,olist_orders_dataset.csv,True
6,products,olist_products_dataset.csv,True
7,sellers,olist_sellers_dataset.csv,True
8,category_translation,product_category_name_translation.csv,True


## 2. Load Raw Data

The raw CSV files are loaded into a dictionary called `dfs`.

The table names follow the same naming used in the data understanding notebook.

In [7]:
dfs = {table_name: pd.read_csv(RAW_DIR / file_name) for table_name, file_name in expected_files.items()}

shape_summary = pd.DataFrame([
    {"table_name": table_name, "rows": df.shape[0], "columns": df.shape[1]}
    for table_name, df in dfs.items()
]).sort_values("rows", ascending=False).reset_index(drop=True)

display(shape_summary)

,table_name,rows,columns
0,geolocation,1000163,5
1,order_items,112650,7
2,order_payments,103886,5
3,customers,99441,5
4,orders,99441,8
5,order_reviews,99224,7
6,products,32951,9
7,sellers,3095,4
8,category_translation,71,2


## 3. Create Cleaning Copies

The raw DataFrames are preserved in `dfs`.

All cleaning operations are applied to independent copies stored in `clean_dfs`.

In [8]:
clean_dfs = {table_name: df.copy() for table_name, df in dfs.items()}

print(f"Clean copies created for {len(clean_dfs)} tables.")

Clean copies created for 9 tables.


## 4. Column Name Consistency

Column names were already reviewed during the data understanding stage.

The raw Olist tables use consistent `snake_case` naming, so no additional column name standardization is required in this notebook.

Original column names are kept to preserve compatibility with the official dataset documentation and future relational modeling steps.

## 5. Convert Date Columns

Datetime-like columns were identified during the data understanding stage.

In this notebook, those columns are converted from text to `datetime` so they can be used later for delivery analysis, target creation, and time-based feature engineering.

Invalid or missing values are converted to `NaT` using `errors="coerce"`.

In [9]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"]
}

datetime_conversion_summary = []

for table_name, columns in datetime_columns.items():
    df = clean_dfs[table_name]

    for column in columns:
        missing_before = df[column].isna().sum()
        df[column] = pd.to_datetime(df[column], errors="coerce")
        missing_after = df[column].isna().sum()

        datetime_conversion_summary.append({
            "table_name": table_name,
            "column": column,
            "missing_before": missing_before,
            "missing_after": missing_after,
            "new_missing_after_conversion": missing_after - missing_before,
            "dtype_after": df[column].dtype
        })

datetime_conversion_summary = pd.DataFrame(datetime_conversion_summary)

display(datetime_conversion_summary)

,table_name,column,missing_before,missing_after,new_missing_after_conversion,dtype_after
0,orders,order_purchase_timestamp,0,0,0,datetime64[us]
1,orders,order_approved_at,160,160,0,datetime64[us]
2,orders,order_delivered_carrier_date,1783,1783,0,datetime64[us]
3,orders,order_delivered_customer_date,2965,2965,0,datetime64[us]
4,orders,order_estimated_delivery_date,0,0,0,datetime64[us]
5,order_items,shipping_limit_date,0,0,0,datetime64[us]
6,order_reviews,review_creation_date,0,0,0,datetime64[us]
7,order_reviews,review_answer_timestamp,0,0,0,datetime64[us]


All existing date values were converted successfully.

No additional missing values were introduced during datetime conversion.

## 6. Exact Duplicate Removal

Exact duplicate rows are fully repeated records.

Only complete row duplicates are removed at this stage. Business-key duplicates are not removed because repeated keys can be valid in transactional tables.

In [10]:
duplicate_summary = []

for table_name, df in clean_dfs.items():
    duplicate_summary.append({
        "table_name": table_name,
        "rows_before": df.shape[0],
        "exact_duplicate_rows": df.duplicated().sum(),
        "exact_duplicate_pct": df.duplicated().mean()
    })

duplicate_summary = (
    pd.DataFrame(duplicate_summary)
    .sort_values("exact_duplicate_rows", ascending=False)
    .reset_index(drop=True)
)

display(duplicate_summary)

,table_name,rows_before,exact_duplicate_rows,exact_duplicate_pct
0,geolocation,1000163,261831,0.261788
1,customers,99441,0,0.000000
2,order_items,112650,0,0.000000
3,order_payments,103886,0,0.000000
4,order_reviews,99224,0,0.000000
5,orders,99441,0,0.000000
6,products,32951,0,0.000000
7,sellers,3095,0,0.000000
8,category_translation,71,0,0.000000


In [11]:
rows_before_dedup = {table_name: df.shape[0] for table_name, df in clean_dfs.items()}

clean_dfs = {
    table_name: df.drop_duplicates().reset_index(drop=True)
    for table_name, df in clean_dfs.items()
}

dedup_summary = pd.DataFrame([
    {
        "table_name": table_name,
        "rows_before": rows_before_dedup[table_name],
        "rows_after": clean_dfs[table_name].shape[0],
        "rows_removed": rows_before_dedup[table_name] - clean_dfs[table_name].shape[0]
    }
    for table_name in clean_dfs.keys()
]).sort_values("rows_removed", ascending=False).reset_index(drop=True)

display(dedup_summary)

,table_name,rows_before,rows_after,rows_removed
0,geolocation,1000163,738332,261831
1,customers,99441,99441,0
2,order_items,112650,112650,0
3,order_payments,103886,103886,0
4,order_reviews,99224,99224,0
5,orders,99441,99441,0
6,products,32951,32951,0
7,sellers,3095,3095,0
8,category_translation,71,71,0


## 7. Review Table Key Validation

The previous data understanding notebook showed that `review_id` is not unique in the `order_reviews` table.

This notebook checks whether duplicated review IDs represent invalid duplicate rows or valid review records linked to different orders.

In [12]:
review_key_checks = pd.DataFrame([
    {
        "candidate_key": "review_id",
        "unique_values": clean_dfs["order_reviews"][["review_id"]].drop_duplicates().shape[0],
        "duplicate_rows": clean_dfs["order_reviews"].duplicated(subset=["review_id"], keep=False).sum(),
        "missing_key_values": clean_dfs["order_reviews"][["review_id"]].isna().any(axis=1).sum()
    },
    {
        "candidate_key": "review_id + order_id",
        "unique_values": clean_dfs["order_reviews"][["review_id", "order_id"]].drop_duplicates().shape[0],
        "duplicate_rows": clean_dfs["order_reviews"].duplicated(subset=["review_id", "order_id"], keep=False).sum(),
        "missing_key_values": clean_dfs["order_reviews"][["review_id", "order_id"]].isna().any(axis=1).sum()
    }
])

review_key_checks["status"] = np.where(
    (review_key_checks["duplicate_rows"] == 0) & (review_key_checks["missing_key_values"] == 0),
    "OK",
    "Review needed"
)

display(review_key_checks)

,candidate_key,unique_values,duplicate_rows,missing_key_values,status
0,review_id,98410,1603,0,Review needed
1,review_id + order_id,99224,0,0,OK


Duplicated `review_id` values are kept because they are not exact duplicate records.

For future relational modeling, `review_id + order_id` can be used as a reliable compound key for `order_reviews`.

## 8. Missing Value Decisions

Missing values were already reviewed during the data understanding stage.

In this notebook, missing values are handled conservatively. No aggressive imputation is applied because missing values may carry business meaning, especially in delivery-related columns, product attributes, and optional review text fields.

In [13]:
missing_decisions = pd.DataFrame([
    {
        "table_name": "orders",
        "columns": "order_approved_at, order_delivered_carrier_date, order_delivered_customer_date",
        "decision": "Keep missing values",
        "reason": "Missing timestamps may indicate canceled, unavailable, or incomplete orders. These fields are needed for later target creation and delivery analysis."
    },
    {
        "table_name": "order_reviews",
        "columns": "review_comment_title, review_comment_message",
        "decision": "Keep missing values",
        "reason": "Review text fields are optional customer feedback fields. Missing text does not make the review invalid."
    },
    {
        "table_name": "products",
        "columns": "product_category_name, product_name_lenght, product_description_lenght, product_photos_qty",
        "decision": "Keep missing values",
        "reason": "Missing catalog attributes affect a small share of products and can be handled later depending on the analysis or modeling use case."
    },
    {
        "table_name": "products",
        "columns": "product_weight_g, product_length_cm, product_height_cm, product_width_cm",
        "decision": "Keep missing values",
        "reason": "Only 2 products are affected. Rows are kept to avoid unnecessary data loss at this stage."
    }
])

with pd.option_context("display.max_colwidth", None):
    display(missing_decisions)

,table_name,columns,decision,reason
0,orders,"order_approved_at, order_delivered_carrier_date, order_delivered_customer_date",Keep missing values,"Missing timestamps may indicate canceled, unavailable, or incomplete orders. These fields are needed for later target creation and delivery analysis."
1,order_reviews,"review_comment_title, review_comment_message",Keep missing values,Review text fields are optional customer feedback fields. Missing text does not make the review invalid.
2,products,"product_category_name, product_name_lenght, product_description_lenght, product_photos_qty",Keep missing values,Missing catalog attributes affect a small share of products and can be handled later depending on the analysis or modeling use case.
3,products,"product_weight_g, product_length_cm, product_height_cm, product_width_cm",Keep missing values,Only 2 products are affected. Rows are kept to avoid unnecessary data loss at this stage.


## 9. Conservative Text Cleaning

Text fields are cleaned conservatively.

The cleaning is limited to removing leading/trailing whitespace and converting empty strings into missing values.

Identifier columns are not transformed beyond whitespace removal, and no aggressive category standardization is applied at this stage.

In [14]:
text_cleaning_summary = []

for table_name, df in clean_dfs.items():
    text_columns = [
        column for column in df.columns
        if is_object_dtype(df[column]) or is_string_dtype(df[column])
    ]

    for column in text_columns:
        original = df[column].copy()

        cleaned = original.astype("string").str.strip()
        cleaned = cleaned.replace("", pd.NA)

        clean_dfs[table_name][column] = cleaned

        text_cleaning_summary.append({
            "table_name": table_name,
            "column": column,
            "values_changed": (original.fillna("<NA>").astype("string") != cleaned.fillna("<NA>")).sum(),
            "missing_before": original.isna().sum(),
            "missing_after": cleaned.isna().sum()
        })

text_cleaning_summary = pd.DataFrame(text_cleaning_summary)

text_cleaning_changes = (
    text_cleaning_summary
    [(text_cleaning_summary["values_changed"] > 0) | (text_cleaning_summary["missing_after"] > text_cleaning_summary["missing_before"])]
    .sort_values(["table_name", "column"])
    .reset_index(drop=True)
)

if text_cleaning_changes.empty:
    print("No text cleaning changes were needed.")
else:
    display(text_cleaning_changes)

,table_name,column,values_changed,missing_before,missing_after
0,geolocation,geolocation_city,1,0,0
1,order_reviews,review_comment_message,9451,58247,58274
2,order_reviews,review_comment_title,1998,87656,87658


Text cleaning only affected a small number of fields.

Most changes came from review text fields, where leading/trailing whitespace was removed and a few empty strings were converted into missing values. No rows were removed.

## 10. Product Category Translation

The products table contains product categories in Portuguese.

The category translation table is used to add an English category name to the clean products table. This improves readability for later EDA, dashboards, and reporting.

This step does not create advanced features and does not remove products without a translated category.

In [15]:
products = clean_dfs["products"].copy()
category_translation = clean_dfs["category_translation"][["product_category_name", "product_category_name_english"]].drop_duplicates().copy()

if "product_category_name_english" in products.columns:
    products = products.drop(columns=["product_category_name_english"])

products = products.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

clean_dfs["products"] = products

translation_summary = pd.DataFrame([{
    "product_rows": products.shape[0],
    "products_with_category": products["product_category_name"].notna().sum(),
    "products_without_category": products["product_category_name"].isna().sum(),
    "products_with_english_category": products["product_category_name_english"].notna().sum(),
    "products_without_english_category": products["product_category_name_english"].isna().sum(),
    "unique_categories_without_translation": products.loc[
        products["product_category_name"].notna() & products["product_category_name_english"].isna(),
        "product_category_name"
    ].nunique()
}])

display(translation_summary)

,product_rows,products_with_category,products_without_category,products_with_english_category,products_without_english_category,unique_categories_without_translation
0,32951,32341,610,32328,623,2


In [16]:
untranslated_categories = (
    products.loc[
        products["product_category_name"].notna() & products["product_category_name_english"].isna(),
        ["product_category_name"]
    ]
    .value_counts()
    .reset_index(name="product_count")
)

if untranslated_categories.empty:
    print("All non-missing product categories have an English translation.")
else:
    display(untranslated_categories)

,product_category_name,product_count
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


Most products with a Portuguese category were successfully enriched with an English category.

A small number of products still have no English category because their original category is either missing or not present in the translation table.

No products are removed and no manual translations are created at this stage.

## 11. Geolocation Aggregation

The geolocation table contains multiple records per zip code prefix.

For future joins with customers or sellers, using the raw geolocation table directly could multiply rows. To avoid this, an aggregated zip code prefix reference table is created.

The aggregation keeps one row per `geolocation_zip_code_prefix` using:

- median latitude
- median longitude
- most frequent city
- most frequent state
- number of geolocation records behind each prefix

In [17]:
geo = clean_dfs["geolocation"].copy()

geo_prefix_summary = pd.DataFrame([{
    "geolocation_rows_after_dedup": geo.shape[0],
    "unique_zip_code_prefixes": geo["geolocation_zip_code_prefix"].nunique(),
    "zip_prefixes_with_multiple_rows": (geo.groupby("geolocation_zip_code_prefix").size() > 1).sum(),
    "max_records_per_zip_prefix": geo.groupby("geolocation_zip_code_prefix").size().max(),
    "rows_with_missing_coordinates": geo[["geolocation_lat", "geolocation_lng"]].isna().any(axis=1).sum()
}])

display(geo_prefix_summary)

,geolocation_rows_after_dedup,unique_zip_code_prefixes,zip_prefixes_with_multiple_rows,max_records_per_zip_prefix,rows_with_missing_coordinates
0,738332,19015,17823,779,0


In [18]:
def mode_or_na(series):
    mode_values = series.dropna().mode()
    return mode_values.iloc[0] if not mode_values.empty else pd.NA


geolocation_zip_prefix = (
    geo
    .dropna(subset=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"])
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat_median=("geolocation_lat", "median"),
        geolocation_lng_median=("geolocation_lng", "median"),
        geolocation_city=("geolocation_city", mode_or_na),
        geolocation_state=("geolocation_state", mode_or_na),
        geolocation_records_count=("geolocation_zip_code_prefix", "size")
    )
)

clean_dfs["geolocation_zip_prefix"] = geolocation_zip_prefix

display(geolocation_zip_prefix.head())
print(f"Aggregated geolocation rows: {geolocation_zip_prefix.shape[0]:,}")

,geolocation_zip_code_prefix,geolocation_lat_median,geolocation_lng_median,geolocation_city,geolocation_state,geolocation_records_count
0,1001,-23.549951,-46.634027,sao paulo,SP,11
1,1002,-23.548228,-46.635247,sao paulo,SP,6
2,1003,-23.548977,-46.635313,sao paulo,SP,11
3,1004,-23.549550,-46.634771,sao paulo,SP,14
4,1005,-23.549763,-46.636100,sao paulo,SP,13


Aggregated geolocation rows: 19,015


The raw geolocation table still contains many records per zip code prefix after exact duplicate removal.

The aggregated `geolocation_zip_prefix` table reduces the data to one row per prefix, from 738,332 rows to 19,015 rows.

This table should be used for future joins with customers or sellers to avoid unintended row multiplication.

## 12. Leakage-Sensitive Columns

Some columns are kept in the clean tables even though they must not be used as predictive features.

This is intentional. Delivery and review-related fields may be needed later for target creation, validation, or post-delivery analysis.

The future machine learning notebook must explicitly exclude post-delivery variables from the feature matrix.

In [19]:
leakage_sensitive_columns = pd.DataFrame([
    {
        "table_name": "orders",
        "column": "order_delivered_customer_date",
        "reason": "Needed to create the future target, but only known after delivery.",
        "future_ml_use": "Exclude from features"
    },
    {
        "table_name": "orders",
        "column": "order_delivered_carrier_date",
        "reason": "Logistics event that may not be known at prediction time depending on the use case.",
        "future_ml_use": "Review carefully before modeling"
    },
    {
        "table_name": "orders",
        "column": "order_status",
        "reason": "Final or near-final order status can leak delivery outcome information.",
        "future_ml_use": "Exclude or restrict depending on prediction timing"
    },
    {
        "table_name": "order_reviews",
        "column": "review_score",
        "reason": "Customer review information is created after the order experience.",
        "future_ml_use": "Exclude from features"
    },
    {
        "table_name": "order_reviews",
        "column": "review_comment_title, review_comment_message",
        "reason": "Review text is post-delivery customer feedback.",
        "future_ml_use": "Exclude from features"
    },
    {
        "table_name": "order_reviews",
        "column": "review_creation_date, review_answer_timestamp",
        "reason": "Review timestamps occur after the customer experience.",
        "future_ml_use": "Exclude from features"
    }
])

with pd.option_context("display.max_colwidth", None):
    display(leakage_sensitive_columns)

,table_name,column,reason,future_ml_use
0,orders,order_delivered_customer_date,"Needed to create the future target, but only known after delivery.",Exclude from features
1,orders,order_delivered_carrier_date,Logistics event that may not be known at prediction time depending on the use case.,Review carefully before modeling
2,orders,order_status,Final or near-final order status can leak delivery outcome information.,Exclude or restrict depending on prediction timing
3,order_reviews,review_score,Customer review information is created after the order experience.,Exclude from features
4,order_reviews,"review_comment_title, review_comment_message",Review text is post-delivery customer feedback.,Exclude from features
5,order_reviews,"review_creation_date, review_answer_timestamp",Review timestamps occur after the customer experience.,Exclude from features


## 13. Save Clean Outputs

Cleaned tables are saved locally to `data/processed/`.

This folder is ignored by Git, so these CSV files are local working artifacts and are not uploaded to the public repository.

In [20]:
output_files = {
    "customers": "customers_clean.csv",
    "geolocation": "geolocation_clean.csv",
    "geolocation_zip_prefix": "geolocation_zip_prefix_clean.csv",
    "order_items": "order_items_clean.csv",
    "order_payments": "order_payments_clean.csv",
    "order_reviews": "order_reviews_clean.csv",
    "orders": "orders_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "category_translation": "category_translation_clean.csv"
}

for table_name, output_file in output_files.items():
    clean_dfs[table_name].to_csv(PROCESSED_DIR / output_file, index=False)

saved_outputs = pd.DataFrame([
    {
        "table_name": table_name,
        "output_file": output_file,
        "rows": clean_dfs[table_name].shape[0],
        "columns": clean_dfs[table_name].shape[1]
    }
    for table_name, output_file in output_files.items()
])

display(saved_outputs)

,table_name,output_file,rows,columns
0,customers,customers_clean.csv,99441,5
1,geolocation,geolocation_clean.csv,738332,5
2,geolocation_zip_prefix,geolocation_zip_prefix_clean.csv,19015,6
3,order_items,order_items_clean.csv,112650,7
4,order_payments,order_payments_clean.csv,103886,5
5,order_reviews,order_reviews_clean.csv,99224,7
6,orders,orders_clean.csv,99441,8
7,products,products_clean.csv,32951,10
8,sellers,sellers_clean.csv,3095,4
9,category_translation,category_translation_clean.csv,71,2


## 14. Cleaning Summary

The final cleaning summary documents the main row and column changes applied in this notebook.

For the derived `geolocation_zip_prefix` table, the row reduction represents aggregation by zip code prefix, not row deletion.

In [25]:
cleaning_actions = {
    "customers": "No row-level cleaning required. Text fields were conservatively stripped where applicable.",
    "geolocation": "Exact duplicate rows removed. Text fields were conservatively stripped where applicable.",
    "geolocation_zip_prefix": "Derived table aggregated from geolocation using one row per zip code prefix, with median coordinates and modal city/state.",
    "order_items": "Date column converted to datetime. No rows removed.",
    "order_payments": "No row-level cleaning required. Text fields were conservatively stripped where applicable.",
    "order_reviews": "Date columns converted to datetime. Review text fields stripped and empty strings converted to missing values. Review rows kept despite non-unique review_id.",
    "orders": "Order timestamp columns converted to datetime. Missing delivery-related timestamps kept for later analysis and target creation.",
    "products": "English product category added from translation table. Missing or untranslated categories kept.",
    "sellers": "No row-level cleaning required. Text fields were conservatively stripped where applicable.",
    "category_translation": "Used as reference table for English product category enrichment."
}

cleaning_summary_rows = []

for table_name, output_file in output_files.items():
    if table_name == "geolocation_zip_prefix":
        original_rows = clean_dfs["geolocation"].shape[0]
        original_columns = clean_dfs["geolocation"].shape[1]
        change_type = "Aggregated derived table"
    else:
        original_rows = dfs[table_name].shape[0]
        original_columns = dfs[table_name].shape[1]
        change_type = "Cleaned source table"

    cleaned_rows = clean_dfs[table_name].shape[0]
    cleaned_columns = clean_dfs[table_name].shape[1]

    cleaning_summary_rows.append({
        "table_name": table_name,
        "output_file": output_file,
        "original_rows": original_rows,
        "cleaned_rows": cleaned_rows,
        "rows_removed_or_aggregated": original_rows - cleaned_rows,
        "original_columns": original_columns,
        "cleaned_columns": cleaned_columns,
        "change_type": change_type,
        "key_cleaning_actions": cleaning_actions[table_name]
    })

cleaning_summary = pd.DataFrame(cleaning_summary_rows)

display(
    cleaning_summary.style.set_properties(
        **{"white-space": "nowrap"}
    )
)

,table_name,output_file,original_rows,cleaned_rows,rows_removed_or_aggregated,original_columns,cleaned_columns,change_type,key_cleaning_actions
0,customers,customers_clean.csv,99441,99441,0,5,5,Cleaned source table,No row-level cleaning required. Text fields were conservatively stripped where applicable.
1,geolocation,geolocation_clean.csv,1000163,738332,261831,5,5,Cleaned source table,Exact duplicate rows removed. Text fields were conservatively stripped where applicable.
2,geolocation_zip_prefix,geolocation_zip_prefix_clean.csv,738332,19015,719317,5,6,Aggregated derived table,"Derived table aggregated from geolocation using one row per zip code prefix, with median coordinates and modal city/state."
3,order_items,order_items_clean.csv,112650,112650,0,7,7,Cleaned source table,Date column converted to datetime. No rows removed.
4,order_payments,order_payments_clean.csv,103886,103886,0,5,5,Cleaned source table,No row-level cleaning required. Text fields were conservatively stripped where applicable.
5,order_reviews,order_reviews_clean.csv,99224,99224,0,7,7,Cleaned source table,Date columns converted to datetime. Review text fields stripped and empty strings converted to missing values. Review rows kept despite non-unique review_id.
6,orders,orders_clean.csv,99441,99441,0,8,8,Cleaned source table,Order timestamp columns converted to datetime. Missing delivery-related timestamps kept for later analysis and target creation.
7,products,products_clean.csv,32951,32951,0,9,10,Cleaned source table,English product category added from translation table. Missing or untranslated categories kept.
8,sellers,sellers_clean.csv,3095,3095,0,4,4,Cleaned source table,No row-level cleaning required. Text fields were conservatively stripped where applicable.
9,category_translation,category_translation_clean.csv,71,71,0,2,2,Cleaned source table,Used as reference table for English product category enrichment.


In [26]:
cleaning_summary.to_csv(PROCESSED_DIR / "cleaning_summary.csv", index=False)

print(f"Cleaning summary saved to: {PROCESSED_DIR / 'cleaning_summary.csv'}")

Cleaning summary saved to: C:\Users\johan\Desktop\supplyguard-delivery-risk\data\processed\cleaning_summary.csv


## 15. Conclusions

This notebook created clean and consistent processed versions of the raw Olist tables.

Main cleaning outcomes:

- Date columns were converted to `datetime`.
- Exact duplicate rows were removed from the geolocation table.
- The known `order_reviews.review_id` issue was reviewed and documented.
- Missing values were handled conservatively without aggressive imputation.
- Text fields were cleaned with minimal transformations.
- English product category names were added to the products table.
- A zip code prefix-level geolocation reference table was created to avoid row multiplication in future joins.
- Leakage-sensitive columns were kept for future target creation and analysis, but documented as unsafe for direct ML feature use.
- Clean outputs were saved locally to `data/processed/`.

The cleaned tables are now ready for the next project phase: relational modeling and SQL preparation.